<a href="https://colab.research.google.com/github/wnstj1126-debug/-/blob/main/M0_BRIDGE_2048_SOURCE_LOCK_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# M0_BRIDGE_2048_SOURCE_LOCK_v1

**목적**: Bridge가 읽을 FEMTO 원본 CSV의 정확한 목록·순서·해시·구조를 잠근다.

| 항목 | 값 |
|:---|:---|
| Source Lock 버전 | M0_BRIDGE_2048_SOURCE_LOCK_v1 |
| 대상 축 | H축 (COL_H=4) |
| 스냅샷 샘플 수 | 2560 samples (0.1s × 25600 Hz) |
| 감사 선행 조건 | AUDIT_PASS or AUDIT_PASS_WITH_CRITERIA_WARNING |

## 절대 원칙 (Read-only)

- 원본 CSV 수정·삭제·이동·이름 변경 **금지**
- 중복 의심 파일 자동 삭제 **금지**
- M0 기준 파일 (`frozen.json`, `legacy.csv`, 기준 노트북) 변경 **금지**
- M0 feature 재계산·Mahalanobis 재계산·threshold 변경 **금지**
- 2560→2048 변환·M2 학습 **금지** (이 단계에서)
- 모든 출력은 `bridge_outputs/YYYYMMDD_HHMMSS_source_lock_v1/` 에만 기록

## 기준 CSV 수 (Learning)

| Bearing | 기준 N |
|:---|---:|
| Bearing1_1 | 2803 |
| Bearing1_2 | 871 |
| Bearing2_1 | 911 |
| Bearing2_2 | 797 |
| Bearing3_1 | 515 |
| Bearing3_2 | 1637 |
| **합계** | **7534** |

In [1]:
# ================================================================
# 셀 01 — 설정 및 공통 함수
# ================================================================

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import hashlib, json, math, os, re, shutil, traceback
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd

# ── 고정 설정 ────────────────────────────────────────────────
SOURCE_LOCK_VERSION = "M0_BRIDGE_2048_SOURCE_LOCK_v1"

FEMTO_FS                    = 25600
FEMTO_SNAPSHOT_DURATION_SEC = 0.1
FEMTO_DECISION_INTERVAL_SEC = 10.0
COL_H                       = 4
EXPECTED_SNAPSHOT_SAMPLES   = 2560   # 25600 Hz × 0.1 s

LEARNING_EXPECTED = {
    "Bearing1_1": 2803,
    "Bearing1_2": 871,
    "Bearing2_1": 911,
    "Bearing2_2": 797,
    "Bearing3_1": 515,
    "Bearing3_2": 1637,
}
LEARNING_EXPECTED_TOTAL = sum(LEARNING_EXPECTED.values())  # 7534

TEST_EXPECTED = {
    "Bearing1_3": 590,  "Bearing1_4": 1139, "Bearing1_5": 2302,
    "Bearing1_6": 2232, "Bearing1_7": 1502, "Bearing2_3": 1202,
    "Bearing2_4": 612,  "Bearing2_5": 2002, "Bearing2_6": 572,
    "Bearing2_7": 172,  "Bearing3_3": 1802,
}

FULL_TEST_EXPECTED = {
    "Bearing1_3": 2375, "Bearing1_4": 1428, "Bearing1_5": 2463,
    "Bearing1_6": 2448, "Bearing1_7": 2259, "Bearing2_3": 1955,
    "Bearing2_4": 751,  "Bearing2_5": 2311, "Bearing2_6": 701,
    "Bearing2_7": 230,  "Bearing3_3": 434,
}

SPLIT_DIR_NAMES = {
    "LEARNING":  "Training(Learning)_set",
    "TEST":      "Test(Test)_set",
    "FULL_TEST": "Validation(Full_Test)_Set",
}

SPLIT_EXPECTED = {
    "LEARNING":  LEARNING_EXPECTED,
    "TEST":      TEST_EXPECTED,
    "FULL_TEST": FULL_TEST_EXPECTED,
}

PROJECT_ROOT_CANDIDATES = [
    Path("/content/drive/MyDrive/Colab Notebooks/field_iis3dwb"),
    Path("/content/drive/My Drive/Colab Notebooks/field_iis3dwb"),
]

FEMTO_ROOT_CANDIDATES = [
    Path("/content/drive/MyDrive/Colab Notebooks/3.RunToFailure_Raw_Data/PRONOSTIA_FEMTO_Bearing"),
    Path("/content/drive/My Drive/Colab Notebooks/3.RunToFailure_Raw_Data/PRONOSTIA_FEMTO_Bearing"),
    Path("/content/drive/MyDrive/Colab Notebooks/RunToFailure_Raw_Data/PRONOSTIA_FEMTO_Bearing"),
    Path("/content/drive/My Drive/Colab Notebooks/RunToFailure_Raw_Data/PRONOSTIA_FEMTO_Bearing"),
]

REFERENCE_SOURCE_FILES = [
    "BASELINE_M0_frozen.json",
    "m0_baseline_result.csv",
]

REFERENCE_NOTEBOOKS = [
    "M0_FEMTO_Baseline_v1_baseline\uace0\uc815.ipynb",
    "M0_FEMTO_Baseline_v1_baseline.ipynb",
    "M0_FEMTO_Baseline_v1.ipynb",
]

# ── assert ───────────────────────────────────────────────────
assert FEMTO_FS == 25600
assert FEMTO_SNAPSHOT_DURATION_SEC == 0.1
assert COL_H == 4
assert EXPECTED_SNAPSHOT_SAMPLES == int(FEMTO_FS * FEMTO_SNAPSHOT_DURATION_SEC)
assert LEARNING_EXPECTED_TOTAL == 7534

# ── 공통 함수 ────────────────────────────────────────────────
def now_iso():
    return datetime.now().isoformat(timespec="seconds")

def natural_key(s):
    return [int(t) if t.isdigit() else t.lower()
            for t in re.split(r"(\d+)", str(s))]

def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            c = f.read(chunk)
            if not c:
                break
            h.update(c)
    return h.hexdigest()

def file_meta(path):
    st = Path(path).stat()
    return {
        "absolute_path": str(Path(path).resolve()),
        "file_name":     Path(path).name,
        "file_size":     int(st.st_size),
        "mtime_ns":      int(st.st_mtime_ns),
        "sha256":        sha256_file(path),
    }

def first_existing(candidates):
    for p in candidates:
        if Path(p).exists():
            return Path(p)
    return None

def write_json(path, obj):
    def _default(v):
        if isinstance(v, Path): return str(v)
        if isinstance(v, np.integer): return int(v)
        if isinstance(v, np.floating):
            return None if np.isnan(v) else float(v)
        if isinstance(v, np.bool_): return bool(v)
        if isinstance(v, set): return sorted(v)
        try:
            if pd.isna(v): return None
        except Exception:
            pass
        return str(v)
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2, default=_default)

def is_valid_csv_candidate(path: Path) -> tuple:
    """CSV 원본 후보 여부와 제외 사유 반환."""
    name = path.name
    if not path.is_file():
        return False, "NOT_A_FILE"
    if path.suffix.lower() != ".csv":
        return False, f"WRONG_SUFFIX:{path.suffix}"
    if name.startswith(".") or name.startswith("~"):
        return False, "TEMP_FILE_PREFIX"
    if ".ipynb_checkpoints" in str(path):
        return False, "IPYNB_CHECKPOINT"
    if path.stat().st_size == 0:
        return False, "EMPTY_FILE"
    return True, ""

print(f"[SETUP] {SOURCE_LOCK_VERSION} 초기화 완료")
print(f"  FEMTO_FS={FEMTO_FS} Hz, COL_H={COL_H}, EXPECTED_SNAPSHOT={EXPECTED_SNAPSHOT_SAMPLES} samples")
print(f"  Learning 기준 총 CSV: {LEARNING_EXPECTED_TOTAL}")
print("  모든 assert 통과 ✅")

Mounted at /content/drive
[SETUP] M0_BRIDGE_2048_SOURCE_LOCK_v1 초기화 완료
  FEMTO_FS=25600 Hz, COL_H=4, EXPECTED_SNAPSHOT=2560 samples
  Learning 기준 총 CSV: 7534
  모든 assert 통과 ✅


In [2]:
# ================================================================
# 셀 02 — 경로 결정 및 출력 폴더 생성
# ================================================================

PROJECT_ROOT = first_existing(PROJECT_ROOT_CANDIDATES)
if PROJECT_ROOT is None:
    raise FileNotFoundError("PROJECT_ROOT를 찾지 못했습니다.")

FEMTO_ROOT = first_existing(FEMTO_ROOT_CANDIDATES)
if FEMTO_ROOT is None:
    raise FileNotFoundError("FEMTO_ROOT를 찾지 못했습니다.")

RUN_ID     = datetime.now().strftime("%Y%m%d_%H%M%S") + "_source_lock_v1"
OUTPUT_DIR = PROJECT_ROOT / "bridge_outputs" / RUN_ID
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 70)
print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"FEMTO_ROOT   : {FEMTO_ROOT}")
print(f"OUTPUT_DIR   : {OUTPUT_DIR}")
print("=" * 70)

PROJECT_ROOT : /content/drive/MyDrive/Colab Notebooks/field_iis3dwb
FEMTO_ROOT   : /content/drive/MyDrive/Colab Notebooks/3.RunToFailure_Raw_Data/PRONOSTIA_FEMTO_Bearing
OUTPUT_DIR   : /content/drive/MyDrive/Colab Notebooks/field_iis3dwb/bridge_outputs/20260725_092158_source_lock_v1


In [3]:
# ================================================================
# 셀 03 — 감사 Gate 확인
# ================================================================

ALLOWED_AUDIT_STATUSES   = {"AUDIT_PASS", "AUDIT_PASS_WITH_CRITERIA_WARNING"}
ALLOWED_NEXT_ACTIONS     = {"READY_FOR_M0_BRIDGE_2048"}

# 최신 FINAL_AUDIT_SUMMARY_v1_1.json 탐색
audit_summary_candidates = sorted(
    list(PROJECT_ROOT.glob("audit_outputs/*/FINAL_AUDIT_SUMMARY_v1_1.json")),
    key=lambda p: p.stat().st_mtime,
    reverse=True,
)

AUDIT_GATE_PASS = False
audit_gate_info = {"found": False, "path": None, "reason": "NOT_FOUND"}

if not audit_summary_candidates:
    print("⚠️  FINAL_AUDIT_SUMMARY_v1_1.json 없음")
    audit_gate_info["reason"] = "AUDIT_SUMMARY_NOT_FOUND"
else:
    audit_path = audit_summary_candidates[0]
    try:
        with open(audit_path, "r", encoding="utf-8") as f:
            audit_data = json.load(f)

        final_status = audit_data.get("final_status", "")
        next_action  = audit_data.get("recommended_next_action", "")
        src_unchanged = audit_data.get("source_files_unchanged", False)

        AUDIT_GATE_PASS = (
            final_status in ALLOWED_AUDIT_STATUSES
            and next_action in ALLOWED_NEXT_ACTIONS
        )

        audit_gate_info = {
            "found":                  True,
            "path":                   str(audit_path),
            "sha256":                 sha256_file(audit_path),
            "final_status":           final_status,
            "recommended_next_action": next_action,
            "source_files_unchanged": src_unchanged,
            "gate_pass":              AUDIT_GATE_PASS,
            "reason":                 "OK" if AUDIT_GATE_PASS else "UNACCEPTABLE_STATUS",
        }
    except Exception as e:
        audit_gate_info["reason"] = f"PARSE_ERROR:{e}"

write_json(OUTPUT_DIR / "audit_gate_check.json", audit_gate_info)

print(f"[GATE] audit_gate_pass    = {AUDIT_GATE_PASS}")
print(f"       final_status       = {audit_gate_info.get('final_status', 'N/A')}")
print(f"       next_action        = {audit_gate_info.get('recommended_next_action', 'N/A')}")

if not AUDIT_GATE_PASS:
    print("\n🚨 SOURCE_LOCK_BLOCKED_BY_AUDIT_GATE")
    print("   감사 조건이 충족되지 않았습니다. 이후 셀은 참고용으로만 실행하십시오.")

[GATE] audit_gate_pass    = True
       final_status       = AUDIT_PASS_WITH_CRITERIA_WARNING
       next_action        = READY_FOR_M0_BRIDGE_2048


In [4]:
# ================================================================
# 셀 04 — 기준 파일 SHA-256 사전 기록
# ================================================================

ref_snapshot_before = {}

# 기준 파일 (frozen JSON, legacy CSV)
for fname in REFERENCE_SOURCE_FILES:
    candidates = sorted(PROJECT_ROOT.rglob(fname),
                        key=lambda p: p.stat().st_mtime, reverse=True)
    if candidates:
        p = candidates[0]
        ref_snapshot_before[str(p.resolve())] = {"type": "M0_REFERENCE", **file_meta(p)}
        print(f"  [REF] {p.name}: {sha256_file(p)[:16]}...")
    else:
        print(f"  [REF] {fname}: MISSING")

# 기준 노트북
for fname in REFERENCE_NOTEBOOKS:
    candidates = sorted(PROJECT_ROOT.rglob(fname),
                        key=lambda p: p.stat().st_mtime, reverse=True)
    if candidates:
        p = candidates[0]
        ref_snapshot_before[str(p.resolve())] = {"type": "M0_NOTEBOOK", **file_meta(p)}
        print(f"  [NB]  {p.name}: {sha256_file(p)[:16]}...")
        break  # 첫 번째만

# 최신 audit summary
if audit_gate_info.get("path"):
    p = Path(audit_gate_info["path"])
    ref_snapshot_before[str(p.resolve())] = {"type": "AUDIT_SUMMARY", **file_meta(p)}

print(f"\n[REF SNAPSHOT] {len(ref_snapshot_before)}개 기준 파일 기록 완료")

  [REF] BASELINE_M0_frozen.json: a4632ad1b9c51976...
  [REF] m0_baseline_result.csv: 63f493e6e3482157...
  [NB]  M0_FEMTO_Baseline_v1.ipynb: 921f1bbea7b460d7...

[REF SNAPSHOT] 4개 기준 파일 기록 완료


In [5]:
# ================================================================
# 셀 05 — 확장자별 현황 조사
# ================================================================

ext_rows = []

for split_key, split_dir_name in SPLIT_DIR_NAMES.items():
    split_dir = FEMTO_ROOT / split_dir_name
    if not split_dir.exists():
        print(f"  [WARN] {split_dir_name} 없음")
        continue

    bearing_dirs = sorted(
        [d for d in split_dir.iterdir()
         if d.is_dir() and re.fullmatch(r"Bearing\d+_\d+", d.name)],
        key=lambda d: natural_key(d.name),
    )

    for bd in bearing_dirs:
        record_uid = f"{split_key}/{bd.name}"
        all_files  = [f for f in bd.iterdir() if f.is_file()]

        # 확장자별 집계
        ext_map = {}
        for f in all_files:
            ext = f.suffix.lower() or "(no_ext)"
            if ext not in ext_map:
                ext_map[ext] = {"count": 0, "total_bytes": 0,
                                "files": []}
            ext_map[ext]["count"] += 1
            ext_map[ext]["total_bytes"] += f.stat().st_size
            ext_map[ext]["files"].append(f.name)

        csv_count     = ext_map.get(".csv", {}).get("count", 0)
        non_csv_count = sum(v["count"] for k, v in ext_map.items() if k != ".csv")

        for ext, info in sorted(ext_map.items()):
            sorted_files = sorted(info["files"], key=natural_key)
            ext_rows.append({
                "split":              split_key,
                "record_uid":         record_uid,
                "logical_bearing_id": bd.name,
                "extension":          ext,
                "file_count":         info["count"],
                "total_size_bytes":   info["total_bytes"],
                "first_file_name":    sorted_files[0] if sorted_files else "",
                "last_file_name":     sorted_files[-1] if sorted_files else "",
                "csv_file_count":     csv_count,
                "non_csv_file_count": non_csv_count,
                "all_file_count":     len(all_files),
            })

EXT_DF = pd.DataFrame(ext_rows)
EXT_DF.to_csv(OUTPUT_DIR / "source_extension_inventory.csv",
              index=False, encoding="utf-8-sig")

print("=== 확장자별 현황 (CSV 이외 파일) ===")
non_csv = EXT_DF[EXT_DF["extension"] != ".csv"]
if len(non_csv) > 0:
    print(non_csv[["record_uid","extension","file_count","total_size_bytes"]].to_string(index=False))
else:
    print("  CSV 외 파일 없음")

print(f"\n저장: {OUTPUT_DIR}/source_extension_inventory.csv")

=== 확장자별 현황 (CSV 이외 파일) ===
  CSV 외 파일 없음

저장: /content/drive/MyDrive/Colab Notebooks/field_iis3dwb/bridge_outputs/20260725_092158_source_lock_v1/source_extension_inventory.csv


In [7]:
# ================================================================
# 셀 06 — CSV 수 기준 비교
# ================================================================

count_rows = []

for split_key, split_dir_name in SPLIT_DIR_NAMES.items():
    split_dir    = FEMTO_ROOT / split_dir_name
    expected_map = SPLIT_EXPECTED[split_key]
    gate_scope   = "REQUIRED_LEARNING" if split_key == "LEARNING" else "REFERENCE_NON_LEARNING"

    if not split_dir.exists():
        continue

    bearing_dirs = sorted(
        [d for d in split_dir.iterdir()
         if d.is_dir() and re.fullmatch(r"Bearing\d+_\d+", d.name)],
        key=lambda d: natural_key(d.name),
    )

    for bd in bearing_dirs:
        record_uid = f"{split_key}/{bd.name}"
        csv_files  = sorted(
            [f for f in bd.iterdir()
             if f.is_file() and f.suffix.lower() == ".csv"
             and not f.name.startswith((".","~"))
             and f.stat().st_size > 0],
            key=lambda f: natural_key(f.name),
        )
        actual   = len(csv_files)
        expected = expected_map.get(bd.name)
        diff     = actual - expected if expected is not None else None
        match    = (diff == 0) if diff is not None else False

        status = "PASS" if match else ("EXTRA" if (diff or 0) > 0 else "MISSING")
        if expected is None:
            status = "NO_REFERENCE"

        count_rows.append({
            "record_uid":          record_uid,
            "split":               split_key,
            "logical_bearing_id":  bd.name,
            "expected_csv_count":  expected,
            "actual_csv_count":    actual,
            "count_difference":    diff,
            "count_match":         match,
            "gate_scope":          gate_scope,
            "status":              status,
        })

COUNT_DF = pd.DataFrame(count_rows)
COUNT_DF.to_csv(OUTPUT_DIR / "split_csv_count_check.csv",
                index=False, encoding="utf-8-sig")

# Learning gate 판정
learn_rows = COUNT_DF[COUNT_DF["split"] == "LEARNING"]
LEARNING_COUNT_PASS = bool(
    len(learn_rows) == 6
    and learn_rows["count_match"].all()
)
LEARNING_ACTUAL_TOTAL = int(learn_rows["actual_csv_count"].sum())

print("=== CSV 수 기준 비교 ===")
display(COUNT_DF[["record_uid","expected_csv_count","actual_csv_count","count_difference","gate_scope","status"]])
print(f"\nLearning 기준 총계: 기대={LEARNING_EXPECTED_TOTAL}, 실제={LEARNING_ACTUAL_TOTAL}")
print(f"LEARNING_COUNT_PASS = {LEARNING_COUNT_PASS}")

=== CSV 수 기준 비교 ===


,record_uid,expected_csv_count,actual_csv_count,count_difference,gate_scope,status
0,LEARNING/Bearing1_1,2803,3269,466,REQUIRED_LEARNING,EXTRA
1,LEARNING/Bearing1_2,871,1015,144,REQUIRED_LEARNING,EXTRA
2,LEARNING/Bearing2_1,911,1062,151,REQUIRED_LEARNING,EXTRA
3,LEARNING/Bearing2_2,797,797,0,REQUIRED_LEARNING,PASS
4,LEARNING/Bearing3_1,515,604,89,REQUIRED_LEARNING,EXTRA
5,LEARNING/Bearing3_2,1637,1637,0,REQUIRED_LEARNING,PASS
6,TEST/Bearing1_3,590,590,0,REFERENCE_NON_LEARNING,PASS
7,TEST/Bearing1_4,1139,1327,188,REFERENCE_NON_LEARNING,EXTRA
8,TEST/Bearing1_5,2302,2677,375,REFERENCE_NON_LEARNING,EXTRA
9,TEST/Bearing1_6,2232,2232,0,REFERENCE_NON_LEARNING,PASS



Learning 기준 총계: 기대=7534, 실제=8384
LEARNING_COUNT_PASS = False


In [8]:
# ================================================================
# 셀 07 — 전체 CSV Source Manifest 생성 (SHA-256 포함)
# ================================================================

import os

CHECKPOINT_PATH = OUTPUT_DIR / "source_hash_checkpoint.csv"

# 체크포인트 로드 (재실행 시 기존 해시 재사용)
chk_cache = {}
if CHECKPOINT_PATH.exists():
    try:
        chk_df = pd.read_csv(CHECKPOINT_PATH, dtype=str)
        for _, row in chk_df.iterrows():
            key = (str(row.get("absolute_path","")),
                   str(row.get("file_size_bytes","")),
                   str(row.get("modified_time_ns","")))
            chk_cache[key] = str(row.get("sha256",""))
        print(f"  체크포인트 로드: {len(chk_cache)}개 캐시 항목")
    except Exception as e:
        print(f"  체크포인트 로드 실패 (무시): {e}")

manifest_rows = []
total_bearings = sum(len(SPLIT_EXPECTED[s]) for s in SPLIT_DIR_NAMES)
done_count = 0

for split_key, split_dir_name in SPLIT_DIR_NAMES.items():
    split_dir = FEMTO_ROOT / split_dir_name
    if not split_dir.exists():
        continue

    bearing_dirs = sorted(
        [d for d in split_dir.iterdir()
         if d.is_dir() and re.fullmatch(r"Bearing\d+_\d+", d.name)],
        key=lambda d: natural_key(d.name),
    )

    for bd in bearing_dirs:
        done_count += 1
        record_uid = f"{split_key}/{bd.name}"

        all_files = sorted(
            [f for f in bd.iterdir() if f.is_file()],
            key=lambda f: natural_key(f.name),
        )

        csv_seq_idx = 0

        for f in all_files:
            valid, reason = is_valid_csv_candidate(f)
            st = f.stat()
            abs_path = str(f.resolve())
            size_b   = int(st.st_size)
            mtime_ns = int(st.st_mtime_ns)

            # SHA-256 (캐시 우선)
            cache_key = (abs_path, str(size_b), str(mtime_ns))
            if valid:
                if cache_key in chk_cache:
                    h = chk_cache[cache_key]
                else:
                    h = sha256_file(f)
                    chk_cache[cache_key] = h
            else:
                h = ""

            seq = csv_seq_idx if valid else None
            if valid:
                csv_seq_idx += 1

            manifest_rows.append({
                "source_lock_version":  SOURCE_LOCK_VERSION,
                "split":               split_key,
                "record_uid":          record_uid,
                "logical_bearing_id":  bd.name,
                "sequence_index":      seq,
                "csv_file_name":       f.name,
                "relative_path":       str(f.relative_to(FEMTO_ROOT)),
                "absolute_path":       abs_path,
                "file_size_bytes":     size_b,
                "modified_time_ns":    mtime_ns,
                "sha256":              h,
                "suffix":              f.suffix.lower(),
                "is_valid_csv_candidate": valid,
                "exclusion_reason":    reason,
                "manifest_status":     "INCLUDED" if valid else "EXCLUDED",
            })

        # 체크포인트 저장 (bearing 단위)
        chk_rows = [
            {"absolute_path": k[0], "file_size_bytes": k[1],
             "modified_time_ns": k[2], "sha256": v}
            for k, v in chk_cache.items()
        ]
        pd.DataFrame(chk_rows).to_csv(CHECKPOINT_PATH, index=False, encoding="utf-8-sig")

        print(f"  [{done_count:2d}/{total_bearings}] {record_uid}: {csv_seq_idx} CSVs")

MANIFEST_DF = pd.DataFrame(manifest_rows)
MANIFEST_DF.to_csv(OUTPUT_DIR / "femto_csv_source_manifest.csv",
                   index=False, encoding="utf-8-sig")

# JSON은 유효 CSV만
valid_manifest = MANIFEST_DF[MANIFEST_DF["is_valid_csv_candidate"]==True].to_dict("records")
write_json(OUTPUT_DIR / "femto_csv_source_manifest.json", valid_manifest)

print(f"\n[MANIFEST] 전체 파일: {len(MANIFEST_DF)}, 유효 CSV: {len(valid_manifest)}")
print(f"저장: femto_csv_source_manifest.csv / .json")

  [ 1/28] LEARNING/Bearing1_1: 3269 CSVs
  [ 2/28] LEARNING/Bearing1_2: 1015 CSVs
  [ 3/28] LEARNING/Bearing2_1: 1062 CSVs
  [ 4/28] LEARNING/Bearing2_2: 797 CSVs
  [ 5/28] LEARNING/Bearing3_1: 604 CSVs
  [ 6/28] LEARNING/Bearing3_2: 1637 CSVs
  [ 7/28] TEST/Bearing1_3: 590 CSVs
  [ 8/28] TEST/Bearing1_4: 1327 CSVs
  [ 9/28] TEST/Bearing1_5: 2677 CSVs
  [10/28] TEST/Bearing1_6: 2232 CSVs
  [11/28] TEST/Bearing1_7: 1752 CSVs
  [12/28] TEST/Bearing2_3: 1202 CSVs
  [13/28] TEST/Bearing2_4: 713 CSVs
  [14/28] TEST/Bearing2_5: 2337 CSVs
  [15/28] TEST/Bearing2_6: 572 CSVs
  [16/28] TEST/Bearing2_7: 200 CSVs
  [17/28] TEST/Bearing3_3: 1860 CSVs
  [18/28] FULL_TEST/Bearing1_3: 2375 CSVs
  [19/28] FULL_TEST/Bearing1_4: 1665 CSVs
  [20/28] FULL_TEST/Bearing1_5: 2873 CSVs
  [21/28] FULL_TEST/Bearing1_6: 2856 CSVs
  [22/28] FULL_TEST/Bearing1_7: 2635 CSVs
  [23/28] FULL_TEST/Bearing2_3: 1955 CSVs
  [24/28] FULL_TEST/Bearing2_4: 876 CSVs
  [25/28] FULL_TEST/Bearing2_5: 2697 CSVs
  [26/28] FULL_TES

In [9]:
# ================================================================
# 셀 08 — 중복 검사 (파일명 / 해시 within-record / cross-split)
# ================================================================

valid_df = MANIFEST_DF[MANIFEST_DF["is_valid_csv_candidate"]==True].copy()

# ── A. 동일 record_uid 내부 파일명 중복 ────────────────────
dup_name_rows = []

for record_uid, grp in valid_df.groupby("record_uid"):
    grp = grp.copy()
    grp["_norm"] = grp["csv_file_name"].str.lower()
    dup_names = grp.groupby("_norm").filter(lambda g: len(g) > 1)

    for norm_name, sub in dup_names.groupby("_norm"):
        dup_name_rows.append({
            "record_uid":          record_uid,
            "normalized_file_name": norm_name,
            "occurrence_count":    len(sub),
            "original_file_names": "|".join(sub["csv_file_name"].tolist()),
            "absolute_paths":      "|".join(sub["absolute_path"].tolist()),
            "status":              "DUPLICATE_FILENAME_WITHIN_RECORD",
        })

DUP_NAME_DF = pd.DataFrame(dup_name_rows)
DUP_NAME_DF.to_csv(OUTPUT_DIR / "csv_duplicate_name_check.csv",
                   index=False, encoding="utf-8-sig")

WITHIN_NAME_DUP_GROUPS = len(DUP_NAME_DF)
print(f"[DUP NAME] within-record 파일명 중복 그룹: {WITHIN_NAME_DUP_GROUPS}")


# ── B. 동일 record_uid 내부 해시 중복 ─────────────────────
dup_hash_within_rows = []

for record_uid, grp in valid_df.groupby("record_uid"):
    dup_hashes = grp.groupby("sha256").filter(lambda g: len(g) > 1)
    for h, sub in dup_hashes.groupby("sha256"):
        dup_hash_within_rows.append({
            "record_uid":       record_uid,
            "sha256":           h,
            "duplicate_count":  len(sub),
            "sequence_indices": "|".join(sub["sequence_index"].astype(str).tolist()),
            "file_names":       "|".join(sub["csv_file_name"].tolist()),
            "absolute_paths":   "|".join(sub["absolute_path"].tolist()),
            "status":           "DUPLICATE_CONTENT_WITHIN_RECORD",
        })

DUP_HASH_WITHIN_DF = pd.DataFrame(dup_hash_within_rows)
DUP_HASH_WITHIN_DF.to_csv(OUTPUT_DIR / "csv_duplicate_hash_within_record.csv",
                           index=False, encoding="utf-8-sig")

WITHIN_HASH_DUP_GROUPS = len(DUP_HASH_WITHIN_DF)
print(f"[DUP HASH] within-record 해시 중복 그룹: {WITHIN_HASH_DUP_GROUPS}")


# ── C. 서로 다른 split 사이 해시 중복 ─────────────────────
dup_cross_rows = []

hash_to_records = valid_df.groupby("sha256")["record_uid"].apply(set)
cross_dup_hashes = hash_to_records[
    hash_to_records.apply(lambda s: len({r.split("/")[0] for r in s}) > 1)
]

UNEXPECTED_LEARNING_CROSS = 0

for h, record_set in cross_dup_hashes.items():
    sub = valid_df[valid_df["sha256"] == h]
    splits_involved = {r.split("/")[0] for r in record_set}

    # TEST ↔ FULL_TEST 사이 = 알려진 현상
    if splits_involved <= {"TEST", "FULL_TEST"}:
        relation  = "EXPECTED_OR_KNOWN_CROSS_SPLIT_OVERLAP"
        blocking  = False
    elif "LEARNING" in splits_involved:
        relation  = "UNEXPECTED_LEARNING_CROSS_SPLIT_OVERLAP"
        blocking  = True
        UNEXPECTED_LEARNING_CROSS += 1
    else:
        relation  = "CROSS_SPLIT_OVERLAP"
        blocking  = False

    dup_cross_rows.append({
        "sha256":           h,
        "occurrence_count": len(sub),
        "record_uids":      "|".join(sorted(record_set)),
        "sequence_indices": "|".join(sub["sequence_index"].astype(str).tolist()),
        "file_names":       "|".join(sub["csv_file_name"].tolist()),
        "relation":         relation,
        "blocking":         blocking,
    })

DUP_CROSS_DF = pd.DataFrame(dup_cross_rows)
DUP_CROSS_DF.to_csv(OUTPUT_DIR / "csv_duplicate_hash_cross_split.csv",
                    index=False, encoding="utf-8-sig")

print(f"[DUP CROSS] cross-split 해시 중복 그룹: {len(DUP_CROSS_DF)}")
print(f"  → UNEXPECTED Learning cross-split: {UNEXPECTED_LEARNING_CROSS}")

[DUP NAME] within-record 파일명 중복 그룹: 0
[DUP HASH] within-record 해시 중복 그룹: 0
[DUP CROSS] cross-split 해시 중복 그룹: 13891
  → UNEXPECTED Learning cross-split: 0


In [10]:
# ================================================================
# 셀 09 — 30개 정밀 구조 probe (Learning 6 × 5)
# ================================================================

probe_rows = []

PROBE_POSITIONS = {
    "first": 0.0,
    "p25":   0.25,
    "mid":   0.5,
    "p75":   0.75,
    "last":  1.0,
}

learn_manifest = (
    MANIFEST_DF[
        (MANIFEST_DF["split"]=="LEARNING") &
        (MANIFEST_DF["is_valid_csv_candidate"]==True)
    ].copy()
)

for bearing_id, grp in learn_manifest.groupby("logical_bearing_id"):
    grp  = grp.sort_values("sequence_index").reset_index(drop=True)
    N    = len(grp)

    for pos_name, ratio in PROBE_POSITIONS.items():
        idx = min(int(round((N-1) * ratio)), N-1)
        row = grp.iloc[idx]
        fpath = Path(row["absolute_path"])

        probe = {
            "record_uid":           row["record_uid"],
            "probe_position":       pos_name,
            "sequence_index":       int(row["sequence_index"]),
            "csv_file_name":        row["csv_file_name"],
            "absolute_path":        row["absolute_path"],
            "row_count":            None,
            "column_count":         None,
            "expected_row_count":   EXPECTED_SNAPSHOT_SAMPLES,
            "row_count_match":      None,
            "selected_column_index": COL_H,
            "selected_column_exists": None,
            "selected_column_numeric": None,
            "nan_count":            None,
            "inf_count":            None,
            "constant_signal":      None,
            "rms":                  None,
            "minimum":              None,
            "maximum":              None,
            "standard_deviation":   None,
            "probe_status":         "PENDING",
            "error_message":        "",
        }

        try:
            df = pd.read_csv(fpath, header=None)
            r, c = df.shape
            probe["row_count"]    = r
            probe["column_count"] = c
            probe["row_count_match"] = (r == EXPECTED_SNAPSHOT_SAMPLES)

            col_exists = c > COL_H
            probe["selected_column_exists"] = col_exists

            if col_exists:
                sig = pd.to_numeric(df.iloc[:, COL_H], errors="coerce")
                numeric_ok = sig.notna().all()
                probe["selected_column_numeric"] = bool(numeric_ok)

                if numeric_ok:
                    arr = sig.values.astype(np.float64)
                    nan_c = int(np.isnan(arr).sum())
                    inf_c = int(np.isinf(arr).sum())
                    probe["nan_count"]        = nan_c
                    probe["inf_count"]        = inf_c
                    probe["constant_signal"]  = bool(np.ptp(arr) == 0)
                    probe["rms"]              = float(np.sqrt(np.mean(arr**2)))
                    probe["minimum"]          = float(arr.min())
                    probe["maximum"]          = float(arr.max())
                    probe["standard_deviation"] = float(arr.std())

                    fail = (
                        not probe["row_count_match"] or
                        c < 5 or
                        not col_exists or
                        not numeric_ok or
                        nan_c > 0 or inf_c > 0 or
                        probe["constant_signal"] or
                        float(arr.std()) == 0
                    )
                    probe["probe_status"] = "FAIL" if fail else "PASS"
                else:
                    probe["probe_status"] = "FAIL"
            else:
                probe["probe_status"] = "FAIL"

        except Exception as e:
            probe["probe_status"]  = "ERROR"
            probe["error_message"] = str(e)

        probe_rows.append(probe)

PROBE_DF = pd.DataFrame(probe_rows)
PROBE_DF.to_csv(OUTPUT_DIR / "csv_structure_probe.csv",
                index=False, encoding="utf-8-sig")

PROBE_TOTAL = len(PROBE_DF)
PROBE_PASS  = int((PROBE_DF["probe_status"] == "PASS").sum())
PROBE_ALL_PASS = (PROBE_PASS == PROBE_TOTAL)

print(f"[PROBE] 정밀 probe {PROBE_TOTAL}개: PASS={PROBE_PASS}, FAIL={PROBE_TOTAL-PROBE_PASS}")
print(f"PROBE_ALL_PASS = {PROBE_ALL_PASS}")
display(PROBE_DF[["record_uid","probe_position","row_count","column_count",
                  "rms","nan_count","probe_status"]])

[PROBE] 정밀 probe 30개: PASS=26, FAIL=4
PROBE_ALL_PASS = False


,record_uid,probe_position,row_count,column_count,rms,nan_count,probe_status
0,LEARNING/Bearing1_1,first,2560,6,0.561746,0.0,PASS
1,LEARNING/Bearing1_1,p25,2560,6,0.313878,0.0,PASS
2,LEARNING/Bearing1_1,mid,2560,6,0.606463,0.0,PASS
3,LEARNING/Bearing1_1,p75,2560,6,1.065547,0.0,PASS
4,LEARNING/Bearing1_1,last,517,5,163.274317,0.0,FAIL
5,LEARNING/Bearing1_2,first,2560,6,0.538710,0.0,PASS
6,LEARNING/Bearing1_2,p25,2560,6,0.338841,0.0,PASS
7,LEARNING/Bearing1_2,mid,2560,6,0.395962,0.0,PASS
8,LEARNING/Bearing1_2,p75,2560,6,0.358869,0.0,PASS
9,LEARNING/Bearing1_2,last,596,1,NaN,NaN,FAIL


In [11]:
# ================================================================
# 셀 10 — Learning 전체 CSV 경량 구조 검사
# ================================================================

SCHEMA_CHECKPOINT = OUTPUT_DIR / "learning_schema_checkpoint.csv"

schema_cache = {}
if SCHEMA_CHECKPOINT.exists():
    try:
        sc_df = pd.read_csv(SCHEMA_CHECKPOINT, dtype=str)
        for _, row in sc_df.iterrows():
            schema_cache[str(row.get("absolute_path",""))] = row.to_dict()
        print(f"  schema checkpoint 로드: {len(schema_cache)}개")
    except Exception as e:
        print(f"  schema checkpoint 로드 실패: {e}")

schema_rows = []

for bearing_id, grp in (
    learn_manifest.groupby("logical_bearing_id")
):
    grp    = grp.sort_values("sequence_index").reset_index(drop=True)
    N_this = len(grp)
    print(f"  {bearing_id}: {N_this} CSVs 검사 중...")

    for _, row in grp.iterrows():
        abs_path = row["absolute_path"]

        # 캐시 확인 (절대경로 + 크기 + mtime)
        cache_hit = schema_cache.get(abs_path)
        if cache_hit and (
            str(cache_hit.get("file_size_bytes","")) == str(row["file_size_bytes"]) and
            str(cache_hit.get("modified_time_ns","")) == str(row["modified_time_ns"])
        ):
            schema_rows.append(cache_hit)
            continue

        sr = {
            "record_uid":          row["record_uid"],
            "sequence_index":      row["sequence_index"],
            "csv_file_name":       row["csv_file_name"],
            "absolute_path":       abs_path,
            "file_size_bytes":     row["file_size_bytes"],
            "modified_time_ns":    row["modified_time_ns"],
            "row_count":           None,
            "column_count":        None,
            "row_count_match":     None,
            "col_h_exists":        None,
            "col_h_numeric":       None,
            "col_h_nan":           None,
            "col_h_inf":           None,
            "schema_status":       "PENDING",
            "error_message":       "",
        }

        try:
            df = pd.read_csv(abs_path, header=None)
            r, c = df.shape
            sr["row_count"]    = r
            sr["column_count"] = c
            sr["row_count_match"] = (r == EXPECTED_SNAPSHOT_SAMPLES)
            col_exists = c > COL_H
            sr["col_h_exists"] = col_exists

            if col_exists:
                sig  = pd.to_numeric(df.iloc[:, COL_H], errors="coerce")
                num_ok = sig.notna().all()
                sr["col_h_numeric"] = bool(num_ok)
                if num_ok:
                    arr = sig.values.astype(np.float64)
                    sr["col_h_nan"] = int(np.isnan(arr).sum())
                    sr["col_h_inf"] = int(np.isinf(arr).sum())

            fail = (
                not sr["row_count_match"] or
                not col_exists or
                not sr.get("col_h_numeric") or
                (sr.get("col_h_nan") or 0) > 0 or
                (sr.get("col_h_inf") or 0) > 0
            )
            sr["schema_status"] = "FAIL" if fail else "PASS"

        except Exception as e:
            sr["schema_status"]  = "ERROR"
            sr["error_message"] = str(e)[:200]

        schema_rows.append(sr)
        schema_cache[abs_path] = sr

    # bearing 단위 checkpoint 저장
    pd.DataFrame(schema_rows).to_csv(
        SCHEMA_CHECKPOINT, index=False, encoding="utf-8-sig"
    )

SCHEMA_DF = pd.DataFrame(schema_rows)
SCHEMA_DF.to_csv(OUTPUT_DIR / "learning_csv_schema_check.csv",
                 index=False, encoding="utf-8-sig")

SCHEMA_TOTAL    = len(SCHEMA_DF)
SCHEMA_PASS_CNT = int((SCHEMA_DF["schema_status"] == "PASS").sum())
SCHEMA_ALL_PASS = (SCHEMA_PASS_CNT == SCHEMA_TOTAL and SCHEMA_TOTAL > 0)

print(f"\n[SCHEMA] Learning 전체 {SCHEMA_TOTAL}개: PASS={SCHEMA_PASS_CNT}, FAIL={SCHEMA_TOTAL-SCHEMA_PASS_CNT}")
print(f"SCHEMA_ALL_PASS = {SCHEMA_ALL_PASS}")
if not SCHEMA_ALL_PASS:
    fail_df = SCHEMA_DF[SCHEMA_DF["schema_status"] != "PASS"]
    print(f"\n실패 파일 ({len(fail_df)}개):")
    display(fail_df[["record_uid","csv_file_name","row_count","column_count","schema_status","error_message"]])

  Bearing1_1: 3269 CSVs 검사 중...
  Bearing1_2: 1015 CSVs 검사 중...
  Bearing2_1: 1062 CSVs 검사 중...
  Bearing2_2: 797 CSVs 검사 중...
  Bearing3_1: 604 CSVs 검사 중...
  Bearing3_2: 1637 CSVs 검사 중...

[SCHEMA] Learning 전체 8384개: PASS=7534, FAIL=850
SCHEMA_ALL_PASS = False

실패 파일 (850개):


,record_uid,csv_file_name,row_count,column_count,schema_status,error_message
2803,LEARNING/Bearing1_1,temp_00001.csv,600,5,FAIL,
2804,LEARNING/Bearing1_1,temp_00002.csv,600,5,FAIL,
2805,LEARNING/Bearing1_1,temp_00003.csv,600,5,FAIL,
2806,LEARNING/Bearing1_1,temp_00004.csv,600,5,FAIL,
2807,LEARNING/Bearing1_1,temp_00005.csv,600,5,FAIL,
...,...,...,...,...,...,...
6742,LEARNING/Bearing3_1,temp_00085.csv,417,1,FAIL,
6743,LEARNING/Bearing3_1,temp_00086.csv,600,1,FAIL,
6744,LEARNING/Bearing3_1,temp_00087.csv,600,1,FAIL,
6745,LEARNING/Bearing3_1,temp_00088.csv,600,1,FAIL,


In [12]:
# ================================================================
# 셀 11 — 원본 불변성 확인
# ================================================================

immut_rows = []
SOURCE_FILES_UNCHANGED = True

# A. 기준 파일 (frozen JSON 등)
for abs_path, before in ref_snapshot_before.items():
    p = Path(abs_path)
    exists_after = p.exists()
    if not exists_after:
        unchanged = False
        SOURCE_FILES_UNCHANGED = False
        immut_rows.append({
            "absolute_path": abs_path, "source_type": before["type"],
            "exists_before": True,  "exists_after": False,
            "size_before":   before["file_size"], "size_after": None,
            "mtime_ns_before": before["mtime_ns"], "mtime_ns_after": None,
            "sha256_before": before["sha256"], "sha256_after": None,
            "unchanged": False, "status": "SOURCE_MODIFIED_OR_MISSING",
        })
        continue

    st_after = p.stat()
    size_after  = int(st_after.st_size)
    mtime_after = int(st_after.st_mtime_ns)

    size_ok  = (size_after == before["file_size"])
    mtime_ok = (mtime_after == before["mtime_ns"])

    if size_ok and mtime_ok:
        sha_after = before["sha256"]  # 변화 없으면 재계산 불필요
        unchanged = True
    else:
        sha_after = sha256_file(p)
        unchanged = (sha_after == before["sha256"])

    if not unchanged:
        SOURCE_FILES_UNCHANGED = False

    immut_rows.append({
        "absolute_path":  abs_path, "source_type": before["type"],
        "exists_before":  True,  "exists_after": True,
        "size_before":    before["file_size"], "size_after": size_after,
        "mtime_ns_before": before["mtime_ns"], "mtime_ns_after": mtime_after,
        "sha256_before":  before["sha256"], "sha256_after": sha_after,
        "unchanged":      unchanged,
        "status":         "UNCHANGED" if unchanged else "SOURCE_MODIFIED_OR_MISSING",
    })

# B. Learning CSV 경량 확인 (크기·mtime 변화 파일만 SHA 재계산)
manifest_before = {
    row["absolute_path"]: row
    for _, row in MANIFEST_DF[
        (MANIFEST_DF["split"]=="LEARNING") &
        (MANIFEST_DF["is_valid_csv_candidate"]==True)
    ].iterrows()
}

for abs_path, before_row in manifest_before.items():
    p = Path(abs_path)
    exists_after = p.exists()
    if not exists_after:
        SOURCE_FILES_UNCHANGED = False
        immut_rows.append({
            "absolute_path": abs_path, "source_type": "LEARNING_CSV",
            "exists_before": True,  "exists_after": False,
            "size_before":   before_row["file_size_bytes"], "size_after": None,
            "mtime_ns_before": before_row["modified_time_ns"], "mtime_ns_after": None,
            "sha256_before": before_row["sha256"], "sha256_after": None,
            "unchanged": False, "status": "SOURCE_MODIFIED_OR_MISSING",
        })
        continue

    st_after    = p.stat()
    size_after  = int(st_after.st_size)
    mtime_after = int(st_after.st_mtime_ns)
    size_ok     = (size_after == int(before_row["file_size_bytes"]))
    mtime_ok    = (mtime_after == int(before_row["modified_time_ns"]))

    if size_ok and mtime_ok:
        sha_after = before_row["sha256"]
        unchanged = True
    else:
        sha_after = sha256_file(p)
        unchanged = (sha_after == before_row["sha256"])

    if not unchanged:
        SOURCE_FILES_UNCHANGED = False

    immut_rows.append({
        "absolute_path":  abs_path, "source_type": "LEARNING_CSV",
        "exists_before":  True, "exists_after": True,
        "size_before":    int(before_row["file_size_bytes"]), "size_after": size_after,
        "mtime_ns_before": int(before_row["modified_time_ns"]), "mtime_ns_after": mtime_after,
        "sha256_before":  before_row["sha256"], "sha256_after": sha_after,
        "unchanged":      unchanged,
        "status":         "UNCHANGED" if unchanged else "SOURCE_MODIFIED_OR_MISSING",
    })

IMMUT_DF = pd.DataFrame(immut_rows)
IMMUT_DF.to_csv(OUTPUT_DIR / "source_file_immutability_check.csv",
                index=False, encoding="utf-8-sig")

changed = IMMUT_DF[~IMMUT_DF["unchanged"]]
print(f"[IMMUTABILITY] 총 {len(IMMUT_DF)}개 파일 확인")
print(f"  source_files_unchanged = {SOURCE_FILES_UNCHANGED}")
if len(changed) > 0:
    print(f"  변경 의심 파일 ({len(changed)}개):")
    display(changed[["absolute_path","source_type","status"]])
else:
    print("  ✅ 변경 없음")

[IMMUTABILITY] 총 8388개 파일 확인
  source_files_unchanged = True
  ✅ 변경 없음


In [14]:
# ================================================================
# 셀 12 — Source-lock 최종 판정
# ================================================================

gate_results = {
    "audit_gate_pass":                    AUDIT_GATE_PASS,
    "learning_6_bearings_exist":          (
        len(COUNT_DF[COUNT_DF["split"]=="LEARNING"]) == 6
    ),
    "learning_csv_count_match_all":       LEARNING_COUNT_PASS,
    "within_record_dup_name_zero":        (WITHIN_NAME_DUP_GROUPS == 0),
    "within_record_dup_hash_zero":        (WITHIN_HASH_DUP_GROUPS == 0),
    "learning_schema_all_pass":           SCHEMA_ALL_PASS,
    "probe_30_all_pass":                  PROBE_ALL_PASS,
    "learning_col_h_all_numeric":         SCHEMA_ALL_PASS,  # schema check가 COL_H 포함
    "source_files_unchanged":             SOURCE_FILES_UNCHANGED,
}

# 필수 출력 파일 생성 완료 여부 (이 셀 이후에서 확인)
REQUIRED_OUTPUTS_GENERATED = True  # 셀 13에서 재확인

all_gates_pass = all(gate_results.values())

# 판정 우선순위
if not SOURCE_FILES_UNCHANGED:
    source_lock_status  = "SOURCE_LOCK_INVALID_SOURCE_MODIFIED"
    next_action         = "MANUAL_REVIEW_REQUIRED"
elif not AUDIT_GATE_PASS:
    source_lock_status  = "SOURCE_LOCK_BLOCKED_BY_AUDIT_GATE"
    next_action         = "MANUAL_REVIEW_REQUIRED"
elif UNEXPECTED_LEARNING_CROSS > 0:
    source_lock_status  = "SOURCE_LOCK_MANUAL_REVIEW_CROSS_SPLIT_OVERLAP"
    next_action         = "MANUAL_REVIEW_REQUIRED"
elif not LEARNING_COUNT_PASS:
    source_lock_status  = "SOURCE_LOCK_HOLD_COUNT_MISMATCH"
    next_action         = "MANUAL_REVIEW_REQUIRED"
elif WITHIN_HASH_DUP_GROUPS > 0 or WITHIN_NAME_DUP_GROUPS > 0:
    source_lock_status  = "SOURCE_LOCK_HOLD_DUPLICATE_CONTENT"
    next_action         = "MANUAL_REVIEW_REQUIRED"
elif not SCHEMA_ALL_PASS or not PROBE_ALL_PASS:
    source_lock_status  = "SOURCE_LOCK_HOLD_SCHEMA_MISMATCH"
    next_action         = "MANUAL_REVIEW_REQUIRED"
else:
    source_lock_status  = "SOURCE_LOCK_PASS"
    next_action         = "READY_FOR_M0_BRIDGE_2048_PREP"

print("=" * 70)
print("Gate 결과:")
for gate, result in gate_results.items():
    icon = "✅" if result else "❌"
    print(f"  {icon} {gate}: {result}")
print("=" * 70)
print(f"SOURCE_LOCK_STATUS : {source_lock_status}")
print(f"NEXT ACTION        : {next_action}")
print("=" * 70)

Gate 결과:
  ✅ audit_gate_pass: True
  ✅ learning_6_bearings_exist: True
  ❌ learning_csv_count_match_all: False
  ✅ within_record_dup_name_zero: True
  ✅ within_record_dup_hash_zero: True
  ❌ learning_schema_all_pass: False
  ❌ probe_30_all_pass: False
  ❌ learning_col_h_all_numeric: False
  ✅ source_files_unchanged: True
SOURCE_LOCK_STATUS : SOURCE_LOCK_HOLD_COUNT_MISMATCH
NEXT ACTION        : MANUAL_REVIEW_REQUIRED


In [15]:
# ================================================================
# 셀 13 — SOURCE_LOCK_SUMMARY 생성
# ================================================================

learn_count_df = COUNT_DF[COUNT_DF["split"]=="LEARNING"]
test_count_df  = COUNT_DF[COUNT_DF["split"]=="TEST"]
full_count_df  = COUNT_DF[COUNT_DF["split"]=="FULL_TEST"]

summary_obj = {
    "source_lock_version":           SOURCE_LOCK_VERSION,
    "source_lock_status":            source_lock_status,
    "recommended_next_action":       next_action,
    "created_at":                    now_iso(),
    "project_root":                  str(PROJECT_ROOT),
    "femto_root":                    str(FEMTO_ROOT),
    "output_dir":                    str(OUTPUT_DIR),
    # Gate
    "audit_gate_pass":               AUDIT_GATE_PASS,
    "audit_final_status":            audit_gate_info.get("final_status"),
    "audit_recommended_next_action": audit_gate_info.get("recommended_next_action"),
    # 데이터 구조
    "learning_bearing_count":        int(len(learn_count_df)),
    "test_bearing_count":            int(len(test_count_df)),
    "full_test_bearing_count":       int(len(full_count_df)),
    # Learning CSV 수
    "learning_expected_csv_count_total": LEARNING_EXPECTED_TOTAL,
    "learning_actual_csv_count_total":   LEARNING_ACTUAL_TOTAL,
    "learning_count_match_all":          LEARNING_COUNT_PASS,
    "learning_per_bearing_counts":       {
        row["logical_bearing_id"]: {
            "expected": row["expected_csv_count"],
            "actual":   row["actual_csv_count"],
            "match":    bool(row["count_match"]),
        }
        for _, row in learn_count_df.iterrows()
    },
    # 스키마
    "learning_schema_pass_all":      SCHEMA_ALL_PASS,
    "learning_schema_total":         SCHEMA_TOTAL,
    "learning_schema_pass_count":    SCHEMA_PASS_CNT,
    # 중복
    "within_record_duplicate_name_groups":  WITHIN_NAME_DUP_GROUPS,
    "within_record_duplicate_hash_groups":  WITHIN_HASH_DUP_GROUPS,
    "unexpected_learning_cross_split_hash_groups": UNEXPECTED_LEARNING_CROSS,
    # Probe
    "probe_file_count":              PROBE_TOTAL,
    "probe_pass_count":              PROBE_PASS,
    # 소스 계약
    "selected_axis":                 "H",
    "selected_column_index":         COL_H,
    "expected_snapshot_samples":     EXPECTED_SNAPSHOT_SAMPLES,
    "source_fs_hz":                  FEMTO_FS,
    "source_snapshot_duration_sec":  FEMTO_SNAPSHOT_DURATION_SEC,
    # 불변성
    "source_files_unchanged":        SOURCE_FILES_UNCHANGED,
    "original_m0_modified":          False,
    "bridge_conversion_performed":   False,
    # 게이트 상세
    "gate_results":                  gate_results,
}

write_json(OUTPUT_DIR / "SOURCE_LOCK_SUMMARY.json", summary_obj)

# ── Markdown 보고서 ──────────────────────────────────────────
def _bm(v):
    return "PASS" if v else "FAIL"

try:
    count_md = COUNT_DF[["record_uid","expected_csv_count","actual_csv_count",
                          "count_difference","gate_scope","status"]].to_markdown(index=False)
except Exception:
    count_md = COUNT_DF.to_string(index=False)

try:
    probe_md = PROBE_DF[["record_uid","probe_position","row_count","column_count",
                          "rms","nan_count","probe_status"]].to_markdown(index=False)
except Exception:
    probe_md = PROBE_DF.to_string(index=False)

md = f"""# {SOURCE_LOCK_VERSION} — SOURCE LOCK SUMMARY

## 최종 결과

| 항목 | 값 |
|:---|:---|
| **SOURCE_LOCK_STATUS** | `{source_lock_status}` |
| **NEXT ACTION** | `{next_action}` |
| 생성 시각 | {now_iso()} |

## Gate 결과

| Gate | 결과 |
|:---|:---:|
| Audit gate pass | {_bm(AUDIT_GATE_PASS)} |
| Learning 6 bearings exist | {_bm(gate_results['learning_6_bearings_exist'])} |
| Learning CSV count match | {_bm(LEARNING_COUNT_PASS)} |
| Within-record dup name 0 | {_bm(WITHIN_NAME_DUP_GROUPS==0)} |
| Within-record dup hash 0 | {_bm(WITHIN_HASH_DUP_GROUPS==0)} |
| Learning schema all pass | {_bm(SCHEMA_ALL_PASS)} |
| 30-file probe all pass | {_bm(PROBE_ALL_PASS)} |
| Source files unchanged | {_bm(SOURCE_FILES_UNCHANGED)} |

## Learning CSV 수 비교

- 기대 합계: {LEARNING_EXPECTED_TOTAL}
- 실제 합계: {LEARNING_ACTUAL_TOTAL}

{count_md}

## 중복 검사

- Within-record 파일명 중복 그룹: {WITHIN_NAME_DUP_GROUPS}
- Within-record 해시 중복 그룹: {WITHIN_HASH_DUP_GROUPS}
- Unexpected Learning cross-split 중복 그룹: {UNEXPECTED_LEARNING_CROSS}

## Learning 전체 스키마 검사

- 총 파일: {SCHEMA_TOTAL}
- PASS: {SCHEMA_PASS_CNT}
- FAIL: {SCHEMA_TOTAL - SCHEMA_PASS_CNT}
- SCHEMA_ALL_PASS: {SCHEMA_ALL_PASS}

## 30개 정밀 Probe

- 총: {PROBE_TOTAL} / PASS: {PROBE_PASS}

{probe_md}

## 소스 계약

- 축: H (COL_H = {COL_H})
- 스냅샷 샘플 수: {EXPECTED_SNAPSHOT_SAMPLES}
- 샘플링 주파수: {FEMTO_FS} Hz
- 스냅샷 길이: {FEMTO_SNAPSHOT_DURATION_SEC} s
- bridge_conversion_performed: False
- original_m0_modified: False

## 원본 불변성

- source_files_unchanged: {SOURCE_FILES_UNCHANGED}
"""

with open(OUTPUT_DIR / "SOURCE_LOCK_SUMMARY.md", "w", encoding="utf-8") as f:
    f.write(md)

print(f"[SUMMARY] SOURCE_LOCK_SUMMARY.json / .md 저장 완료")

[SUMMARY] SOURCE_LOCK_SUMMARY.json / .md 저장 완료


In [17]:
# ================================================================
# 셀 14 — 필수 출력 검증 및 최종 출력
# ================================================================

REQUIRED_OUTPUTS = [
    "audit_gate_check.json",
    "source_extension_inventory.csv",
    "split_csv_count_check.csv",
    "femto_csv_source_manifest.csv",
    "femto_csv_source_manifest.json",
    "source_hash_checkpoint.csv",
    "csv_duplicate_name_check.csv",
    "csv_duplicate_hash_within_record.csv",
    "csv_duplicate_hash_cross_split.csv",
    "csv_structure_probe.csv",
    "learning_csv_schema_check.csv",
    "learning_schema_checkpoint.csv",
    "source_file_immutability_check.csv",
    "SOURCE_LOCK_SUMMARY.md",
    "SOURCE_LOCK_SUMMARY.json",
]

missing = [n for n in REQUIRED_OUTPUTS if not (OUTPUT_DIR / n).exists()]
if missing:
    print(f"⚠️  누락 출력 파일: {missing}")
else:
    print("✅ 모든 필수 출력 파일 생성 완료")

# ── 최종 대형 출력 ────────────────────────────────────────────
print()
print("=" * 70)
print(f"  SOURCE LOCK STATUS      : {source_lock_status}")
print(f"  RECOMMENDED NEXT ACTION : {next_action}")
print("=" * 70)
print(f"  Learning expected CSVs  : {LEARNING_EXPECTED_TOTAL}")
print(f"  Learning actual   CSVs  : {LEARNING_ACTUAL_TOTAL}")
print(f"  Schema failures         : {SCHEMA_TOTAL - SCHEMA_PASS_CNT}")
print(f"  Within-record dup groups: {WITHIN_HASH_DUP_GROUPS}")
print(f"  Unexpected LRN cross    : {UNEXPECTED_LEARNING_CROSS}")
print(f"  Source immutability     : {SOURCE_FILES_UNCHANGED}")
print(f"  Probe pass/total        : {PROBE_PASS}/{PROBE_TOTAL}")
print(f"  Output dir              : {OUTPUT_DIR}")
print("=" * 70)

if source_lock_status != "SOURCE_LOCK_PASS":
    print()
    print("⛔  SOURCE_LOCK_PASS가 아닙니다.")
    print("    Bridge PREP을 자동 실행하지 않습니다.")
    print("    위 결과를 확인한 후 수동으로 조치하십시오.")
else:
    print()
    print("🔒 SOURCE LOCK PASS")
    print("   Bridge 입력 CSV 목록·순서·해시·구조가 잠겼습니다.")
    print("   M0_BRIDGE_2048_PREP_v1으로 진행할 수 있습니다.")

# 원본 변경은 어떤 경우에도 허용하지 않음
if not SOURCE_FILES_UNCHANGED:
    raise RuntimeError(
        "SOURCE_LOCK_INVALID_SOURCE_MODIFIED: 원본 파일 해시가 변경되었습니다."
    )

✅ 모든 필수 출력 파일 생성 완료

  SOURCE LOCK STATUS      : SOURCE_LOCK_HOLD_COUNT_MISMATCH
  RECOMMENDED NEXT ACTION : MANUAL_REVIEW_REQUIRED
  Learning expected CSVs  : 7534
  Learning actual   CSVs  : 8384
  Schema failures         : 850
  Within-record dup groups: 0
  Unexpected LRN cross    : 0
  Source immutability     : True
  Probe pass/total        : 26/30
  Output dir              : /content/drive/MyDrive/Colab Notebooks/field_iis3dwb/bridge_outputs/20260725_092158_source_lock_v1

⛔  SOURCE_LOCK_PASS가 아닙니다.
    Bridge PREP을 자동 실행하지 않습니다.
    위 결과를 확인한 후 수동으로 조치하십시오.
